In [ ]:
import pandas as pd
import os
import re
import json
import numpy as np

In [ ]:
FILE_PATH = "../data/Scenario_6"
TIME_WINDOW = '1S'
OUTPUT_FILENAME = f'../data/scenario_6_aggregated_{TIME_WINDOW}.csv'


def parse_statistics(file_path):
    """
    Parses the Phone_Statistics file with the corrected format:
    TIMESTAMP::... data="{...}"
    """
    stats_records = []
    try:
        with open(file_path, 'r') as f:
            for line in f:
                try:
                    parts = line.split('::', 1)
                    if len(parts) < 2:
                        continue
                    timestamp = float(parts[0])

                    data_prefix = 'data="'
                    json_start_index = parts[1].find(data_prefix)
                    if json_start_index == -1:
                        continue
                    
                    json_string = parts[1][json_start_index + len(data_prefix):-2]
                    json_data = json.loads(json_string)

                    record = {
                        'timestamp': timestamp,
                        'bwe': int(json_data.get('bwe', np.nan)),
                        'buffer_level_ms': int(json_data.get('bh', np.nan)),
                        'video_format_itag': int(json_data.get('fmt', np.nan))
                    }
                    stats_records.append(record)
                except (json.JSONDecodeError, IndexError, ValueError, KeyError):
                    # skip any malformed or incomplete lines
                    continue
        if not stats_records:
            return None
            
        df = pd.DataFrame(stats_records)
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s')
        return df.set_index('timestamp')
    except FileNotFoundError:
        return None

def parse_tcpdump(file_path):
    """
    Parses a Phone_TCPdump file. This regex correctly handles the two-line format
    by only matching the first line containing the timestamp and IP length.
    """
    tcp_pattern = re.compile(
        r'^(?P<timestamp>\d+\.\d+).* length (?P<length>\d+)', re.MULTILINE
    )
    packet_records = []
    try:
        with open(file_path, 'r') as f:
            content = f.read()
            for match in tcp_pattern.finditer(content):
                packet_records.append({
                    'timestamp': float(match.group('timestamp')),
                    'length': int(match.group('length'))
                })
        if not packet_records:
            return None
        df = pd.DataFrame(packet_records)
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s')
        return df
    except FileNotFoundError:
        return None

all_processed_dfs = []
print(f"Starting processing in: {os.path.abspath(FILE_PATH)}")
print(f"Using time window: {TIME_WINDOW}")

for video_id in os.listdir(FILE_PATH):
    video_path = os.path.join(FILE_PATH, video_id)
    if not os.path.isdir(video_path): continue

    for iteration in os.listdir(video_path):
        iteration_path = os.path.join(video_path, iteration)
        if not os.path.isdir(iteration_path): continue
        
        stats_file = next((os.path.join(iteration_path, f) for f in os.listdir(iteration_path) if "Phone_Statistics" in f), None)
        tcpdump_file = next((os.path.join(iteration_path, f) for f in os.listdir(iteration_path) if "Phone_TCPdump" in f), None)

        if not stats_file or not tcpdump_file:
            print(f"    Warning: Missing Statistics or TCPdump file. Skipping.")
            continue
            
        stats_df = parse_statistics(stats_file)
        packets_df = parse_tcpdump(tcpdump_file)

        if stats_df is None or packets_df is None or packets_df.empty:
            print(f"    Warning: Failed to parse data or no packets found. Skipping.")
            continue
            
        packets_df = packets_df.sort_values('timestamp').reset_index(drop=True)
        packets_df['iat_ms'] = packets_df['timestamp'].diff().dt.total_seconds() * 1000
        packets_df = packets_df.set_index('timestamp')

        agg_rules = {
            'length': ['sum', 'mean', 'std'],
            'iat_ms': ['mean', 'std']
        }
        features_df = packets_df.resample(TIME_WINDOW).agg(agg_rules)
        features_df.columns = ['_'.join(col).strip() for col in features_df.columns.values]
        features_df['packet_count'] = packets_df['length'].resample(TIME_WINDOW).count()
        
        features_df.rename(columns={
            'length_sum': 'bytes_sum',
            'length_mean': 'packet_size_mean',
            'length_std': 'packet_size_std',
            'iat_ms_mean': 'iat_mean_ms',
            'iat_ms_std': 'jitter_ms'
        }, inplace=True)
        
        time_window_seconds = pd.to_timedelta(TIME_WINDOW).total_seconds()
        features_df['throughput_kbps'] = (features_df['bytes_sum'] * 8) / (1024 * time_window_seconds)

        merged_df = pd.merge_asof(
            left=features_df.sort_index(),
            right=stats_df.sort_index(),
            left_index=True,
            right_index=True,
            direction='backward'
        )
        
        merged_df['video_id'] = video_id
        merged_df['iteration'] = iteration
        
        all_processed_dfs.append(merged_df)


In [ ]:
if not all_processed_dfs:
    print("\nNo data was processed. The final DataFrame is empty.")
else:
    final_df = pd.concat(all_processed_dfs)
    final_df.reset_index(inplace=True)
    
    final_df.dropna(inplace=True)
    
    # final_df = final_df[[
    #     'timestamp', 'video_id', 'iteration', 
    #     'throughput_kbps', 'packet_count', 'packet_size_mean', 'packet_size_std', 
    #     'iat_mean_ms', 'jitter_ms',
    #     'buffer_level_ms', 'bwe', 'video_format_itag'
    # ]]  
    #video_format has been removed because it is completely useless
    final_df = final_df[[
        'timestamp', 
        'throughput_kbps', 'packet_count', 'packet_size_mean', 'packet_size_std', 
        'iat_mean_ms', 'jitter_ms',
        'buffer_level_ms', 'bwe'
    ]]

    final_df.to_csv(OUTPUT_FILENAME, index=False)

    print(f"\nProcessing complete. Data saved to '{OUTPUT_FILENAME}'")
    print(f"Total rows created: {len(final_df)}")
    print("\nFinal DataFrame Head:")
    print(final_df.head())

In [ ]:
#EDA
df = pd.read_csv('../data/scenario_6_aggregated_1S.csv')

print(df.describe())

print(df.info())

In [ ]:
import matplotlib.pyplot as plt

df.hist(bins=50, figsize=(20, 15))
plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns

sns.scatterplot(data=df, x='throughput_kbps', y='buffer_level_ms')
plt.title('Throughput vs. Buffer Level')
plt.show()

# Scatter plot: How does jitter relate to buffer level?
# We expect to see lower buffer levels when jitter is high.
sns.scatterplot(data=df, x='jitter_ms', y='buffer_level_ms', alpha=0.5)
plt.title('Jitter vs. Buffer Level')
plt.show()

numerical_cols = [
    'throughput_kbps', 'packet_count', 'packet_size_mean', 
    'packet_size_std', 'iat_mean_ms', 'jitter_ms',
    'buffer_level_ms', 'bwe'
]

# correlation matrixx
plt.figure(figsize=(12, 8))
sns.heatmap(df[numerical_cols].corr(), annot=True, cmap='coolwarm')
plt.title('Feature Correlation Matrix')
plt.show()